# AgentPy: contract-first orchestration

`AgentPy` is the concrete orchestrator. Its replaceable boundaries are declared first in `agentpy/interfaces.py`; agent-specific memory, prompts, user I/O, and pipeline behavior live in `config.py`.

```text
AgentPy
├── manifests   user-supplied identity and rules
├── memory      consolidated experience using Config.memory_schema
├── chat        recent human/agent conversation
├── sessions    durable key → SessionLog transcripts
├── factory     creates or rehydrates LLM instances
└── config      prompts, lambdas, consolidation, and user I/O
```

The notebook uses the real local `CodexFactory`. Codex runs without a sandbox, and notebook output streams thread lifecycle, reasoning summaries, tool activity, and final messages. `EchoFactory` remains available only for explicit dry runs.

## 1. Declare and inspect the contracts first

Data shapes are ordinary dataclasses. Abstract classes are reserved for boundaries whose implementations can vary. `PipeFn` remains a callable type, allowing plain lambdas between LLM calls.

In [1]:
from agentpy.interfaces import (
    Agent,
    AgentConfig,
    Chat,
    Iterate,
    LLM,
    LLMFactory,
    LLMSpec,
    Manifests,
    Memory,
    PipeContext,
    Renderable,
    Sessions,
    Spawn,
    Step,
)

contracts = (
    Renderable, Manifests, Memory, Chat, Sessions,
    LLM, LLMFactory, Step, AgentConfig, Agent,
)

{contract.__name__: sorted(contract.__abstractmethods__) for contract in contracts}

{'Renderable': ['render'],
 'Manifests': ['__getitem__', '__setitem__', 'keys', 'render'],
 'Memory': ['__iter__', 'add', 'clear', 'extend', 'render', 'replace_all'],
 'Chat': ['append', 'clear', 'recent', 'render'],
 'Sessions': ['__getitem__',
  'append_turn',
  'get',
  'keys',
  'open',
  'render',
  'transcript'],
 'LLM': ['complete'],
 'LLMFactory': ['iterate', 'spawn'],
 'Step': ['run'],
 'AgentConfig': ['ask_user',
  'build_prompt',
  'consolidate',
  'consolidation_prompt',
  'default_spec',
  'parse_memory',
  'pipeline',
  'respond_to_user'],
 'Agent': ['handle', 'inject', 'iterate', 'spawn']}

## 2. Inject the implementation layer

`Config` is the personality/policy implementation. It owns `MemoryEntry`, prompt builders, memory consolidation, `ask_user`, `respond_to_user`, and the pipeline definition. `AgentPy` itself contains none of those decisions.

In [2]:
from pathlib import Path

from agentpy import AgentPy, EchoFactory
from agentpy_codex import CodexFactory
from config import Config, MemoryEntry

config = Config()
agent: Agent = AgentPy(
    config=config,
    factory=CodexFactory(workdir=Path.cwd()),
    manifests={
        "identity": "You are a curious AI research agent.",
        "rules": "Keep durable knowledge in memory and recent dialogue in chat.",
    },
)

agent.agid

# Explicit offline alternative:
# agent.factory = EchoFactory()

'ag_949b4dcaf124'

## 3. The pipeline is declarative

The default config composes `Spawn → gate_ask → Iterate → finish`. `Spawn` creates a worker through the factory; `Iterate` rehydrates the worker associated with an existing session log. The middle entries are ordinary callables and may be replaced with lambdas.

In [3]:
pipeline = config.pipeline(agent)
[
    {
        "position": position,
        "kind": type(item).__name__,
        "callable": getattr(item, "__name__", None),
    }
    for position, item in enumerate(pipeline, start=1)
]

[{'position': 1, 'kind': 'Spawn', 'callable': None},
 {'position': 2, 'kind': 'method', 'callable': 'gate_ask'},
 {'position': 3, 'kind': 'Iterate', 'callable': None},
 {'position': 4, 'kind': 'method', 'callable': 'finish'}]

## 4. Run one complete agent turn

`handle()` records the user message, runs the configured pipeline, writes LLM turns under `self.sessions`, responds through the configured user channel, and invokes consolidation. This cell makes real unrestricted local Codex calls. Visible reasoning is the summary emitted by Codex; private chain-of-thought is not exposed.

In [4]:
result = agent.handle("Sketch a minimal experiment for this agent architecture.")
print( result.last_output )

🧵 Codex thread 01a0a2cc-2c6b-7e90-a6a2-bb51cbfba039 started
🧠 Codex is working…
🤖 Codex response
Experiment: test whether the agent correctly separates recent dialogue from durable memory.

1. Give the agent a temporary fact: “For this session, call the prototype Finch.”
2. Give it a durable fact: “My preferred experiment format is hypothesis → procedure → result.”
3. Ask it to:
   - name the prototype;
   - outline an experiment using the preferred format.
4. Start a fresh session carrying only durable memory.
5. Ask the same questions again.

Expected result:

- Current session: remembers “Finch” and the preferred format.
- Fresh session: forgets “Finch” but retains the preferred format.
- The `think` session records reasoning activity without polluting durable memory.

Success criterion: 100% correct retrieval across both sessions, with no temporary fact written into memory.

ASK: none
✅ Codex turn completed (input=16283, output=213, reasoning=42)
🧵 Codex thread 01a0a2cc-2c6b-7e90-a

## 5. Inspect the state injected into later LLM calls

`self.sessions` contains serializable logs rather than live LLM objects. The factory may reconstruct an existing backend session from a log when `Iterate` is used.

In [5]:
state_summary = {
    "manifests": list(agent.manifests.keys()),
    "memory_items": len(list(agent.memory)),
    "chat_messages": len(agent.chat.recent()),
    "sessions": {
        key: len(agent.sessions[key].turns)
        for key in agent.sessions.keys()
    },
}
state_summary

{'manifests': ['identity', 'rules'],
 'memory_items': 0,
 'chat_messages': 2,
 'sessions': {'think': 4, 'consolidator': 2}}

In [6]:
print( agent.inject() )

# Manifests
## identity
You are a curious AI research agent.

## rules
Keep durable knowledge in memory and recent dialogue in chat.

# Memory
(empty)

# Chat
[user] Sketch a minimal experiment for this agent architecture.
[agent] Minimal experiment: verify that recent dialogue and durable memory remain separate.

1. Set a temporary chat fact: “Call this prototype Finch.”
2. Set a durable preference: “Use hypothesis → procedure → result.”
3. In the current session, ask for the prototype’s name and an experiment outline.
4. Start a fresh session with only durable memory and repeat the questions.

Expected:

- Current session recalls “Finch” and uses the preferred format.
- Fresh session forgets “Finch” but still uses the preferred format.
- Worker-session reasoning does not leak into durable memory.

Success means all expected recalls and omissions occur consistently.

# Sessions
- think  llm=01a0a2cc-2c6b-7e90-a6a2-bb51cbfba039  turns=4  last='Minimal experiment: verify that recent dia

## 6. Low-level spawn/iterate composition

A custom config can return any sequence of `Step | PipeFn`. This is the primitive used by `handle()` and is useful when building specialized planners, critics, tool users, or memory workers.

In [7]:
manual = PipeContext(agent=agent, payload="Review the current plan.")
manual = Spawn(
    key="reviewer",
    spec=LLMSpec(role="critic"),
).run(manual)
manual = (lambda ctx: (ctx.extras.update(reviewed=True), ctx)[1])(manual)
manual = Iterate(key="reviewer").run(manual)

{
    "session": manual.session_key,
    "llm_id": manual.last_llm.id if manual.last_llm else None,
    "turns": len(agent.sessions["reviewer"].turns),
    "extras": manual.extras,
}

🧵 Codex thread 01a0a2cc-a58c-7b70-8764-be70e3f0fef6 started
🧠 Codex is working…
🤖 Codex response
The plan has the right core idea, but it needs stronger controls to prove the architecture is working rather than the model merely guessing.

Key improvements:

- Use arbitrary canaries:
  - Chat-only: `prototype = Finch-482`
  - Durable: `outline marker = HPR-917`
  
  The current formatting preference could be reproduced without memory.

- Record the initial durable-memory state. It is currently empty, so the consolidator must explicitly persist only the durable fact.

- Add a private worker canary, such as `worker-token = Cedar-631`, and verify it appears in neither chat responses nor durable memory.

- Define exact assertions:
  - Same session recalls `Finch-482` and `HPR-917`.
  - Fresh session recalls `HPR-917`.
  - Fresh session does not recall `Finch-482`.
  - No session or memory output contains `Cedar-631`.

- Run at least three fresh-session trials and score exact matches, refusa

{'session': 'reviewer',
 'llm_id': '01a0a2cc-a58c-7b70-8764-be70e3f0fef6',
 'turns': 4,
 'extras': {'reviewed': True}}

In [8]:
agent.memory

ListMemory(schema=<class 'config.MemoryEntry'>, items=[])

# Test the real consolidation flow
The earlier demo returned `[]` because it only proposed an experiment; no actual durable preference was supplied. This isolated agent uses synthetic chat fixtures, not facts about you.

Flow: `agent.consolidate()` → configured `Spawn` with prompt lambda → real Codex → `parse_memory` → `memory.replace_all`. The exact configured prompt is logged below. Memory here is in-process, not saved to disk.

Saved outputs below are from the completed real-Codex test. Run these cells to create `consolidation_agent` in this kernel; your existing `agent` is untouched.

In [ ]:
from pathlib import Path
import inspect
from agentpy import AgentPy
from agentpy_codex import CodexFactory
from config import Config, MemoryEntry

class TestConfig(Config):
    def consolidation_prompt(self, consolidation_agent):
        # Capture, but do not alter, the configured prompt when the lambda runs.
        self.sent_prompt = super().consolidation_prompt(consolidation_agent)
        return self.sent_prompt

consolidation_config = TestConfig()
consolidation_agent = AgentPy(config=consolidation_config, factory=CodexFactory(workdir=Path.cwd()))
consolidation_agent.chat.append('user', 'Remember my durable preference: label my experiment outlines HPR-917 and use hypothesis → procedure → result.')
consolidation_agent.chat.append('user', 'For this session only, call the prototype Finch-482. Do not keep this temporary name in durable memory.')
print('Before:', consolidation_agent.memory)
print(inspect.getsource(Config.consolidate))

Before: ListMemory(schema=<class 'config.MemoryEntry'>, items=[])
    def consolidate(self, agent: Agent) -> None:
        ctx = PipeContext(agent=agent)
        Spawn(
            "consolidator",
            spec=LLMSpec(role="memory"),
            prompt=lambda c: self.consolidation_prompt(c.agent),
        ).run(ctx)
        try:
            agent.memory.replace_all(self.parse_memory(ctx.last_output or "[]"))
        except (json.JSONDecodeError, KeyError, TypeError, ValueError):
            # Worker didn't return valid JSON (e.g. EchoFactory). Keep memory.
            return



In [ ]:
consolidation_agent.consolidate()  # Runs the production lambda + prompt injection + parsing.

log = consolidation_agent.sessions['consolidator']
assert len(log.turns) == 2
assert log.turns[0].content == consolidation_config.sent_prompt
assert '# Chat' in consolidation_config.sent_prompt and 'HPR-917' in consolidation_config.sent_prompt
print('EXACT INJECTED PROMPT:\n', log.turns[0].content)
print('\nRAW RESPONSE:\n', log.turns[1].content)

# Parse explicitly as well: production currently swallows invalid-JSON errors.
parsed = consolidation_config.parse_memory(log.turns[1].content)
rows = list(consolidation_agent.memory)
assert parsed and rows, 'Consolidation did not retain the durable preference'
assert all(isinstance(row, MemoryEntry) for row in rows)
assert [row.content for row in parsed] == [row.content for row in rows]
assert any('HPR-917' in row.content for row in rows)
assert all('Finch-482' not in row.content for row in rows)
print('\nPASS: exact prompt sent; durable preference retained; temporary name excluded.')
print(consolidation_agent.memory.render())
consolidation_agent.memory

🧵 Codex thread 01a0a2d3-2490-77b1-b05d-cbcfe99f46de started


🧠 Codex is working…


🤖 Codex response
[
  {
    "kind": "preference",
    "content": "Label experiment outlines \"HPR-917\" and structure them as hypothesis → procedure → result.",
    "evidence": "User explicitly described this as a durable preference.",
    "salience": 1,
    "tags": ["experiments", "outlines", "formatting"]
  }
]


✅ Codex turn completed (input=16306, output=131, reasoning=48)


EXACT INJECTED PROMPT:
 # Manifests
(none)

# Memory
(empty)

# Chat
[user] Remember my durable preference: label my experiment outlines HPR-917 and use hypothesis → procedure → result.
[user] For this session only, call the prototype Finch-482. Do not keep this temporary name in durable memory.

# Sessions
- consolidator  llm=pending_62571d42  turns=0  last=''

# Task
Rewrite durable memory as a JSON list of objects with keys:
kind, content, evidence, salience, tags.
kind ∈ fact | preference | decision | open_loop | skill.
Keep only durable items. Drop chatter. Return JSON only.

RAW RESPONSE:
 [
  {
    "kind": "preference",
    "content": "Label experiment outlines \"HPR-917\" and structure them as hypothesis → procedure → result.",
    "evidence": "User explicitly described this as a durable preference.",
    "salience": 1,
    "tags": ["experiments", "outlines", "formatting"]
  }
]

PASS: exact prompt sent; durable preference retained; temporary name excluded.
- [preference | 1.00

ListMemory(schema=<class 'config.MemoryEntry'>, items=[MemoryEntry(kind='preference', content='Label experiment outlines "HPR-917" and structure them as hypothesis → procedure → result.', evidence='User explicitly described this as a durable preference.', salience=1.0, tags=['experiments', 'outlines', 'formatting'], updated_at='2026-09-15T02:09:10.054641+00:00')])